In [ ]:
from google.colab import files
files.upload() # Select your kaggle.json here

# This moves the file to the hidden folder Kaggle expects
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [1]:
# 1. Clean the folder one last time
!rm -rf /content/wikiart_baroque
!mkdir -p /content/wikiart_baroque


In [ ]:
!ls /content/wikiart_baroque | head -n 5

wikiart.zip


In [ ]:
# 1. Install the specialized hub library
!pip install kagglehub -q

import kagglehub

# 2. Download specifically the WikiArt dataset
# This library is 'smarter' about finding the files than the standard CLI
path = kagglehub.dataset_download("steubk/wikiart")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'wikiart' dataset.
Path to dataset files: /kaggle/input/wikiart


In [ ]:
import os

# Create a symbolic link so your code can find the images easily
# The dataset is likely nested as /kaggle/input/wikiart/Baroque
source_path = "/kaggle/input/wikiart/Baroque"

if os.path.exists(source_path):
    !ln -s /kaggle/input/wikiart/Baroque /content/wikiart_baroque
    print("✅ Baroque folder linked successfully!")
    !ls /content/wikiart_baroque | head -n 5
else:
    # If it's nested differently, this finds it
    print("Searching for Baroque folder...")
    !find /kaggle/input/wikiart -name "Baroque" -type d

✅ Baroque folder linked successfully!
Baroque


In [ ]:
!pip install kagglehub -q
import kagglehub
import os

# Download again (Kagglehub is fast)
path = kagglehub.dataset_download("steubk/wikiart")

# Link it immediately so the script can see it
!rm -rf /content/wikiart_baroque
!ln -s "{path}/Baroque" /content/wikiart_baroque

100%|██████████| 31.4G/31.4G [06:38<00:00, 84.5MB/s]

Extracting files...


In [ ]:
import os
image_count = len(os.listdir('/content/wikiart_baroque'))
print(f"✅ Success! {image_count} images found in the linked folder.")

✅ Success! 4240 images found in the linked folder.


In [ ]:
# 1. Remove any hidden zip/tar files from the download
!find /root/.cache/kagglehub -name "*.zip" -delete
!find /root/.cache/kagglehub -name "*.tar.gz" -delete

# 2. Clear the pip cache (where the models were stored during install)
!rm -rf /root/.cache/pip

# 3. Check your available space
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   76G   38G  67% /


In [ ]:
!ls /content/wikiart_baroque | head -n 7

adriaen-brouwer_a-boor-asleep.jpg
adriaen-brouwer_drinkers-in-the-yard.jpg
adriaen-brouwer_dune-landscape-by-moonlight.jpg
adriaen-brouwer_farmers-fight-party.jpg
adriaen-brouwer_father-s-of-unpleasant-duties-1631.jpg
adriaen-brouwer_feeling.jpg
adriaen-brouwer_fumatore.jpg


In [ ]:
import torch
from PIL import Image
import os, json
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel, BlipProcessor, BlipForConditionalGeneration
from google.colab import files

# --- 1. SETUP ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

class BaroqueDataset(Dataset):
    def __init__(self, image_dir):
        self.image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir)
                            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        return {"image": Image.open(path).convert("RGB"), "id": os.path.basename(path)}

# --- 2. EXECUTION ---
dataset = BaroqueDataset("/content/wikiart_baroque")
loader = DataLoader(dataset, batch_size=8, collate_fn=lambda x: x)

CHUNK_SIZE = 500
current_results = []
chunk_count = 1

clip_model.eval()
blip_model.eval()

print(f"🚀 Starting chunked processing of {len(dataset)} images...")

with torch.no_grad():
    for i, batch in enumerate(loader):
        images = [item["image"] for item in batch]
        ids = [item["id"] for item in batch]

        # CLIP Features
        inputs = clip_processor(images=images, return_tensors="pt", padding=True).to(device)
        output_obj = clip_model.get_image_features(**inputs)
        features = getattr(output_obj, 'image_embeds', output_obj[0])
        embeddings = (features / features.norm(dim=-1, keepdim=True)).cpu().numpy()

        # BLIP Captions
        b_inputs = blip_processor(images=images, return_tensors="pt").to(device)
        cap_ids = blip_model.generate(**b_inputs, max_new_tokens=20)
        captions = blip_processor.batch_decode(cap_ids, skip_special_tokens=True)

        for j in range(len(ids)):
            current_results.append({
                "image_id": ids[j],
                "factual_caption": captions[j],
                "clip_embedding": embeddings[j].tolist()
            })

        # Save Checkpoint every 500 images
        if len(current_results) >= CHUNK_SIZE:
            filename = f'member1_baroque_part_{chunk_count}.json'
            with open(filename, 'w') as f:
                json.dump(current_results, f)
            print(f"✅ Chunk {chunk_count} saved: {filename}")

            # Reset for next chunk
            current_results = []
            chunk_count += 1

    # Save any leftover images
    if current_results:
        filename = f'member1_baroque_part_{chunk_count}.json'
        with open(filename, 'w') as f:
            json.dump(current_results, f)
        print(f"✅ Final chunk {chunk_count} saved!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

🚀 Starting chunked processing of 4240 images...
✅ Chunk 1 saved: member1_baroque_part_1.json
✅ Chunk 2 saved: member1_baroque_part_2.json
✅ Chunk 3 saved: member1_baroque_part_3.json
✅ Chunk 4 saved: member1_baroque_part_4.json
✅ Chunk 5 saved: member1_baroque_part_5.json
✅ Chunk 6 saved: member1_baroque_part_6.json
✅ Chunk 7 saved: member1_baroque_part_7.json
✅ Chunk 8 saved: member1_baroque_part_8.json
✅ Final chunk 9 saved!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import shutil
import os

# 1. Define the destination folder on your Google Drive
drive_folder = '/content/drive/MyDrive/Nlp_Project/Processed_data_1'

# 2. Create the folder if it doesn't exist yet
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"✅ Created folder: {drive_folder}")

# 3. Find and move all your part_X.json files
for i in range(1, 10):  # This targets part_1, part_2, part_3, part_4
    filename = f'member1_baroque_part_{i}.json'

    if os.path.exists(filename):
        # Move the file from Colab storage to Drive storage
        shutil.move(filename, os.path.join(drive_folder, filename))
        print(f"🚚 Moved {filename} to Google Drive!")
    else:
        print(f"⚠️ Could not find {filename} in the sidebar.")

print("\n✨ ALL DONE! Check your Google Drive now.")

⚠️ Could not find member1_baroque_part_8.json in the sidebar.
🚚 Moved member1_baroque_part_9.json to Google Drive!

✨ ALL DONE! Check your Google Drive now.


In [ ]:
# 1. Compress the file
!zip member1_results.zip member1_fast_baroque.json

# 2. Download the smaller zip file
from google.colab import files
files.download('member1_results.zip')